# 00 — RSU Assembly

Load all 12 pilot RSU JSON files. Each RSU contains 4-5 native staple foods covering macronutrient categories:
**carb | fat | protein | aromatic | fermented**

Metabolite profiles are per food, not per RSU.

**Null = unmeasured (unknown).** `confirmed_absent` = genuinely not present in the food. These are different.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
from rsu_loader import load_all_rsus, build_food_matrix

In [2]:
rsus = load_all_rsus()
print(f"Loaded {len(rsus)} RSUs")
for rsu in rsus:
    cats = [f.macronutrient_category for f in rsu.staple_foods]
    absent = rsu.absent_categories
    print(f"  {rsu.region_id} {rsu.name}: {cats}" + (f" | absent: {absent}" if absent else ""))

Loaded 12 RSUs
  RSU-01 Boreal Finland: ['carb', 'fat', 'protein', 'aromatic', 'fermented']
  RSU-02 Boreal Canada: ['carb', 'fat', 'protein', 'aromatic', 'fermented']
  RSU-03 Siberian Taiga: ['carb', 'fat', 'protein', 'aromatic', 'fermented']
  RSU-04 Atlantic Western Europe: ['carb', 'fat', 'protein', 'aromatic', 'fermented']
  RSU-05 Central Europe: ['carb', 'fat', 'protein', 'aromatic', 'fermented']
  RSU-06 East Asian Temperate: ['carb', 'fat', 'protein', 'aromatic', 'fermented']
  RSU-07 North American Prairie: ['carb', 'fat', 'protein'] | absent: ['aromatic', 'fermented']
  RSU-08 West African Transition Zone: ['carb', 'fat', 'protein', 'aromatic', 'fermented']
  RSU-09 Amazon Basin: ['carb', 'fat', 'protein', 'aromatic', 'fermented']
  RSU-10 Andean Highlands: ['carb', 'fat', 'protein', 'aromatic', 'fermented']
  RSU-11 South Indian Monsoon: ['carb', 'fat', 'protein', 'aromatic', 'fermented']
  RSU-12 Maritime Southeast Asia: ['carb', 'fat', 'protein', 'aromatic', 'fermented']

## Food matrix — one row per (RSU, food)

In [3]:
df = build_food_matrix(rsus)
print(f"Shape: {df.shape}")
df

Shape: (58, 21)


,region_id,rsu_name,food_name,category,fdc_id,n_confirmed_absent,primary_metabolites.starch_content,primary_metabolites.lipid_content,primary_metabolites.protein_content,organic_acids.lactic_acid,...,organic_acids.malic_acid,primary_metabolites.glucose_concentration,key_flavor_bioactives.tannin_content,organic_acids.acetic_acid,umami_compounds.glutamate,terpenes.pinene,terpenes.linalool,terpenes.limonene,key_flavor_bioactives.capsaicinoids,terpenes.myrcene
0,RSU-01,Boreal Finland,rye bread,carb,2512375,5,88.5810,50.9540,54.1975,-0.45,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RSU-01,Boreal Finland,Atlantic salmon,fat,2684441,4,NaN,56.5550,60.1595,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,RSU-01,Boreal Finland,pike,protein,173680,4,NaN,50.3450,59.6300,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,RSU-01,Boreal Finland,cloudberries,aromatic,169802,3,54.3000,50.4000,51.2000,NaN,...,-0.35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RSU-01,Boreal Finland,fermented Baltic herring,fermented,175118,4,54.8200,59.0000,57.0950,NaN,...,NaN,53.855,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,RSU-02,Boreal Canada,bannock,carb,168058,6,66.9400,50.4900,51.4450,NaN,...,NaN,57.110,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,RSU-02,Boreal Canada,moose,fat,174344,4,NaN,50.3700,61.1200,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,RSU-02,Boreal Canada,walleye,protein,2685576,3,54.3935,50.1510,50.8440,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,RSU-02,Boreal Canada,wild blueberries,aromatic,2346411,3,57.2860,50.1530,50.3515,NaN,...,-0.25,NaN,-100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,RSU-02,Boreal Canada,smoked whitefish,fermented,173712,5,NaN,50.4650,61.7000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Metabolite field coverage — which fields exist across foods

In [4]:
meta_cols = [c for c in df.columns if '.' in c]
print(f"Metabolite dimensions with at least one measurement: {len(meta_cols)}")

coverage = df[meta_cols].notna().sum().sort_values(ascending=False)
coverage = coverage[coverage > 0]
print(coverage.to_string())

Metabolite dimensions with at least one measurement: 15
primary_metabolites.lipid_content            53
primary_metabolites.protein_content          50
primary_metabolites.starch_content           41
umami_compounds.glutamate                    13
primary_metabolites.glucose_concentration     9
organic_acids.lactic_acid                     7
organic_acids.citric_acid                     6
organic_acids.malic_acid                      5
key_flavor_bioactives.tannin_content          3
terpenes.limonene                             3
key_flavor_bioactives.capsaicinoids           3
organic_acids.acetic_acid                     2
terpenes.pinene                               2
terpenes.myrcene                              2
terpenes.linalool                             1


## Coverage by RSU

In [5]:
rsu_coverage = df.groupby('region_id')[meta_cols].apply(lambda x: x.notna().sum().sum())
rsu_coverage = rsu_coverage.sort_values(ascending=False).rename('n_measured_fields')
print(rsu_coverage.to_string())

region_id
RSU-12    22
RSU-05    21
RSU-11    20
RSU-10    19
RSU-04    18
RSU-01    17
RSU-02    17
RSU-03    17
RSU-08    16
RSU-06    15
RSU-09    12
RSU-07     6


## Coverage by food category

In [6]:
cat_coverage = df.groupby('category')[meta_cols].apply(lambda x: x.notna().sum().sum())
cat_coverage = cat_coverage.sort_values(ascending=False).rename('n_measured_fields')
print(cat_coverage.to_string())

category
aromatic     52
carb         48
fermented    37
protein      36
fat          27


## Confirmed absent vs. unmeasured

These are different. `confirmed_absent` = the compound genuinely does not occur in this food.
Null = we do not have a measurement yet.

In [7]:
absent_rows = []
for rsu in rsus:
    for food in rsu.staple_foods:
        for field in food.confirmed_absent:
            absent_rows.append({'region_id': rsu.region_id, 'food': food.name, 'category': food.macronutrient_category, 'confirmed_absent_field': field})

absent_df = pd.DataFrame(absent_rows)
print(f"Total confirmed-absent entries: {len(absent_df)}")

# Which fields are most consistently absent across the pilot set?
absent_df['confirmed_absent_field'].value_counts()

Total confirmed-absent entries: 237


caffeine_concentration       58
theobromine_concentration    58
capsaicinoids                54
linalool                     38
isothiocyanates               9
lactic_acid                   9
myrcene                       6
tannin_content                5
Name: confirmed_absent_field, dtype: int64

## Data collection priority

Foods with fdc_id = null need FDC search. Run `src/fdc_fetcher.py` to populate.

```bash
export FDC_API_KEY=your_key
cd src && python fdc_fetcher.py
```

In [8]:
no_fdc = df[df['fdc_id'].isna()][['region_id', 'food_name', 'category']].copy()
print(f"Foods without FDC ID: {len(no_fdc)}")
no_fdc

Foods without FDC ID: 0


,region_id,food_name,category
